# Cohort retention: dựng heatmap và đọc nó theo ba hướng

Head of CRM hỏi: **"Khách mới có quay lại không? Khách mới năm nay có tốt hơn năm ngoái không?"**. Module 03 đã
dựng bảng cohort retention bằng SQL. Bài này dựng bằng pandas và tập trung vào phần khó hơn: **đọc** bảng đó.

- Ma trận cohort × tuổi `k` (định nghĩa dùng chung của track: cohort là tháng của hoá đơn hợp lệ đầu tiên;
  retention tháng `k` là tỉ lệ khách của cohort có ít nhất một hoá đơn hợp lệ ở tháng thứ `k` sau tháng đầu).
  Ô chưa quan sát được để trống, khác 0.
- **Calendar view**: cùng con số, xếp theo tháng lịch (`cohort + k`) để thấy mùa vụ chạy theo đường chéo.
- Đường retention trung bình: có trọng số theo cỡ cohort, kèm số cohort phía sau mỗi điểm.

Dữ liệu: Online Retail II (UCI, CC BY 4.0), cả hai sheet, bỏ phần trùng 01-09/12/2010. Tháng 12/2011 chưa đủ nên
tháng cuối dùng là 11/2011. Tải: `uv run python data/download_dataset.py online-retail-ii`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/12-product-customer-analytics/exercises/basic/02-cohort-heatmap-reading -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Viết sẵn, không cần sửa: đọc Online Retail II (cache parquet) và lọc dòng hợp lệ theo định nghĩa dùng chung
import tempfile
import uuid
from pathlib import Path

CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache trong thư mục tạm, không ghi vào data/raw


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet Online Retail II. Lần đầu đọc xlsx (30-90 giây) rồi cache parquet theo mtime và cỡ tệp."""
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    xlsx = root / "data" / "raw" / "online-retail-ii" / "online_retail_II.xlsx"
    if not xlsx.exists():
        raise FileNotFoundError("Chưa có dữ liệu. Chạy: uv run python data/download_dataset.py online-retail-ii")
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet"
    if cache.exists():
        try:
            return pd.read_parquet(cache)
        except (OSError, ValueError):  # cache hỏng: đọc lại
            pass
    tx = pd.read_excel(xlsx, sheet_name=sheet)
    tx = tx.assign(Invoice=tx["Invoice"].astype(str), StockCode=tx["StockCode"].astype(str),
                   Description=tx["Description"].astype("string"))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{uuid.uuid4().hex}.tmp")
    tx.to_parquet(tmp, index=False)
    tmp.replace(cache)
    return tx


def clean_sales(tx: pd.DataFrame) -> pd.DataFrame:
    """Bỏ hoá đơn huỷ (C...), Quantity <= 0, Price <= 0; thêm revenue = Quantity * Price."""
    s = tx.assign(Invoice=tx["Invoice"].astype(str))
    s = s[~s.Invoice.str.startswith("C") & (s.Quantity > 0) & (s.Price > 0)].copy()
    s["revenue"] = s.Quantity * s.Price
    return s

## 1. Bảng hoạt động và ma trận retention

`cohort_activity(sales, last) -> DataFrame`: `sales` là dòng hợp lệ (đã qua `clean_sales`). Bỏ dòng không có
`Customer ID`. Trả **một dòng cho mỗi khách × tháng có mua**, cột theo thứ tự `customer_id`, `month`
(`Period` tháng), `cohort` (tháng mua đầu tiên của khách, tính trên **toàn bộ** `sales`), `k` (số tháng từ
`cohort` tới `month`, kiểu `int`). Chỉ giữ `month <= last`; khách có cohort sau `last` vì thế cũng không còn dòng nào.

`retention_matrix(act, last) -> DataFrame`: index là cohort, cột `k` là **mọi** số nguyên từ 0 tới tuổi của cohort
sớm nhất (`last - cohort sớm nhất`), kể cả `k` không có ai mua. Giá trị = số khách của cohort có dòng ở tuổi `k` / cỡ
cohort (số khách ở `k = 0`). Ô có `k` lớn hơn tuổi của cohort (`last - cohort`) là **chưa quan sát được**: `NaN`.
Ô quan sát được mà không ai mua là `0.0`.

In [ ]:
def cohort_activity(sales: pd.DataFrame, last: pd.Period) -> pd.DataFrame:
    """Một dòng mỗi khách × tháng có mua: customer_id, month, cohort, k (chỉ month <= last)."""
    # TODO
    raise NotImplementedError


def retention_matrix(act: pd.DataFrame, last: pd.Period) -> pd.DataFrame:
    """Cohort × k (0..tuổi cohort sớm nhất); NaN ở ô chưa quan sát được."""
    # TODO: đếm khách theo (cohort, k), chia cỡ cohort, đủ cột k, che ô k > tuổi
    raise NotImplementedError

## 2. Calendar view và đường retention trung bình

`calendar_view(matrix) -> DataFrame`: cùng giá trị của `matrix` nhưng **bỏ cột `k = 0`** và đổi cột sang tháng
lịch `cohort + k` (`Period`). Index cohort, cột là các tháng lịch có ít nhất một ô quan sát được, sắp tăng dần. Ô
không có giá trị là `NaN`.

`retention_curve(matrix, sizes, min_cohorts=3) -> DataFrame`: index `k`, ba cột theo thứ tự:

- `weighted`: trung bình các ô quan sát được ở cột `k`, **trọng số là cỡ cohort** (`sizes`, index là cohort):
  trả lời "một khách mới bất kỳ quay lại bao nhiêu %";
- `simple`: trung bình thường của các ô quan sát được;
- `cohorts`: số cohort quan sát được ở `k`.

Điểm có ít hơn `min_cohorts` cohort thì `weighted` và `simple` là `NaN` (vẫn giữ dòng và `cohorts`).

In [ ]:
def calendar_view(matrix: pd.DataFrame) -> pd.DataFrame:
    """Cohort × tháng lịch (cohort + k), bỏ k = 0."""
    # TODO: stack -> bỏ NaN -> cột month = cohort + k -> pivot
    raise NotImplementedError


def retention_curve(matrix: pd.DataFrame, sizes: pd.Series, min_cohorts: int = 3) -> pd.DataFrame:
    """Theo k: weighted (trọng số cỡ cohort), simple, cohorts; NaN khi < min_cohorts cohort."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc hai sheet, bỏ phần trùng, dựng ma trận
LAST = pd.Period("2011-11", "M")                                   # tháng đủ cuối cùng
old, new = read_sheet("Year 2009-2010"), read_sheet("Year 2010-2011")
sales = clean_sales(pd.concat([old[old.InvoiceDate < "2010-12-01"], new], ignore_index=True))
act = cohort_activity(sales, LAST)
sizes = act[act.k == 0].groupby("cohort").size()
matrix = retention_matrix(act, LAST)
print(f"{len(sizes)} cohort, {sizes.sum():,} khách; cohort nhỏ nhất {sizes.idxmin()} ({sizes.min()}), "
      f"lớn nhất {sizes.idxmax()} ({sizes.max()})")
print(matrix.iloc[:8, :8].round(3).to_string())

In [ ]:
# explore: heatmap cohort × k (bỏ k = 0 để cột đó không chiếm thang màu), ghi cỡ cohort bên phải
fig, ax = plt.subplots(figsize=(11, 7))
data = matrix.loc[:, 1:]
im = ax.imshow(data.to_numpy(), cmap="Blues", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(len(data.columns)), data.columns, fontsize=7)
ax.set_yticks(range(len(data)), [f"{c}  (n={sizes[c]})" for c in data.index], fontsize=7)
ax.set_xlabel("tháng thứ k sau tháng mua đầu")
fig.colorbar(im, ax=ax, label="retention (ô trắng: chưa quan sát được)")
ax.set_title("Retention theo cohort: rơi ở tháng 1 rồi đi ngang; cohort 12/2009 cao bất thường", loc="left")
fig.tight_layout()
plt.show()

In [ ]:
# explore: calendar view, mùa vụ chạy thành cột dọc (trước đây là đường chéo)
cal = calendar_view(matrix)
fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(cal.to_numpy(), cmap="Blues", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(len(cal.columns)), [str(c) for c in cal.columns], rotation=90, fontsize=7)
ax.set_yticks(range(len(cal)), [str(c) for c in cal.index], fontsize=7)
ax.set_xlabel("tháng lịch")
fig.colorbar(im, ax=ax, label="retention")
ax.set_title("Cùng con số theo tháng lịch: mọi cohort cùng sáng ở 9-11 và cùng tối ở 12-2", loc="left")
fig.tight_layout()
plt.show()

w = cal.notna().mul(sizes, axis=0)
by_month = cal.fillna(0).mul(sizes, axis=0).sum() / w.sum()
print("retention trung bình (trọng số) theo tháng lịch:", by_month.round(3).to_dict())

In [ ]:
# explore: đường trung bình có trọng số vs đơn giản, kèm số cohort phía sau mỗi điểm
curve = retention_curve(matrix, sizes, min_cohorts=1)
cut = retention_curve(matrix, sizes)                       # min_cohorts = 3
print(curve.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ks = curve.index[1:]
ax.plot(ks, curve.weighted[1:] * 100, color="#bbbbbb", ls="--", label="có trọng số, mọi điểm")
ax.plot(ks, cut.weighted[1:] * 100, color="#4c78a8", marker="o", ms=3, label="có trọng số, ≥ 3 cohort")
ax.plot(ks, cut.simple[1:] * 100, color="#f58518", marker="o", ms=3, label="trung bình đơn giản, ≥ 3 cohort")
for k in ks:
    ax.annotate(str(curve.cohorts[k]), (k, 2), ha="center", fontsize=7, color="#666")
ax.set_ylim(0, 45)
ax.set_xlabel("tháng thứ k (số nhỏ: số cohort phía sau điểm)")
ax.set_ylabel("% khách còn mua")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Đường đi lên ở cuối là do còn 1-2 cohort, không phải khách quay lại", loc="left")
plt.show()

no_dec09 = retention_curve(matrix.drop(pd.Period("2009-12", "M")), sizes.drop(pd.Period("2009-12", "M")))
print("bỏ cohort 12/2009 (cắt trái):", no_dec09.weighted.loc[[1, 3, 6, 12]].round(3).to_dict())

In [ ]:
# explore: so cohort cùng tháng của hai năm (trừ mùa vụ), cỡ cohort bên cạnh
pairs = [(f"2010-{m:02d}", f"2011-{m:02d}") for m in range(1, 11)]
rows = []
for a, b in pairs:
    pa, pb = pd.Period(a, "M"), pd.Period(b, "M")
    rows.append({"tháng": a[-2:], "n_2010": sizes[pa], "m1_2010": matrix.loc[pa, 1], "n_2011": sizes[pb],
                 "m1_2011": matrix.loc[pb, 1], "m3_2010": matrix.loc[pa, 3], "m3_2011": matrix.loc[pb, 3]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

## Đọc heatmap, rồi làm trắc nghiệm

1. **Theo hàng**: cohort 01/2010 rơi bao nhiêu ở tháng 1, rồi đi ngang hay tiếp tục rơi?
2. **Theo cột**: cohort nào có retention tháng 1 thấp nhất? Tháng 1 của cohort đó rơi vào tháng lịch nào?
3. **Theo đường chéo / calendar view**: tháng lịch nào mọi cohort cùng tối đi? Đó là chất lượng cohort hay mùa vụ?
4. **Cohort 12/2009**: vì sao cao hơn hẳn mọi cohort khác? Có nên để nó trong đường trung bình không?
5. **Đường trung bình**: vì sao đường "mọi điểm" đi lên ở cuối? Điểm cuối có bao nhiêu cohort phía sau?
6. **Năm nay so với năm ngoái**: so cohort cùng tháng của 2010 và 2011 ở m1, m3. Viết một câu trả lời cho Head of
   CRM có cả retention lẫn cỡ cohort.

Xong thì mở app, làm phần **trắc nghiệm** của bài.